# skdr-eval — Pairwise / autoscaling quickstart

Pairwise OPE for client-operator decisions with day-varying
eligibility. Use this notebook when your action set is
*per-decision-context*, not fixed across the whole log.

👉 [Open in Colab](https://colab.research.google.com/github/dgenio/skdr-eval/blob/main/examples/notebooks/02_pairwise_quickstart.ipynb)

## 1. Install

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet skdr-eval scikit-learn

## 2. Synthesize pairwise logs

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

import skdr_eval

logs_df, op_daily_df = skdr_eval.make_pairwise_synth(
    n_days=3, n_clients_day=200, n_ops=6, seed=0
)
skdr_eval.validate_pairwise_inputs(logs_df, op_daily_df, metric_col="service_time")
len(logs_df), len(op_daily_df)

## 3. Fit a candidate routing model

In [ ]:
feature_cols = [c for c in logs_df.columns if c.startswith(("cli_", "op_"))]
candidate = HistGradientBoostingRegressor(max_iter=80, random_state=0)
candidate.fit(logs_df[feature_cols].values, logs_df["service_time"].values)

## 4. Evaluate

In [ ]:
artifact = skdr_eval.evaluate_pairwise_models(
    logs_df=logs_df,
    op_daily_df=op_daily_df,
    models={"HGB": candidate},
    metric_col="service_time",
    task_type="regression",
    direction="min",
    strategy="stream_topk",
    topk=4,
    n_splits=2,
    random_state=0,
)
artifact.report[
    ["model", "estimator", "V_hat", "ESS", "match_rate", "support_health"]
].round(4)

## 5. Stakeholder card

In [ ]:
import pathlib

pathlib.Path("artifacts").mkdir(exist_ok=True)
artifact.save_card("artifacts/pairwise_card.html", "HGB")